# 34 RL efficiency phase (A100)

The first RL run (`32_rl`) raised accuracy (dev EX +3.7) but not efficiency: a question teaches efficiency only when the model's correct answers to it differ in cost, which was rare. This phase continues from the RL model kept on validation and trains only where efficiency can be learned:

* **Questions:** 1,953 train questions where a correct query at most 0.9 x gold's cost is known (strict rule: no arbitrary-row picks) or where the RL model's correct answers differed in cost by 10% or more.
* **16 answers per question** (instead of 8): more chances that correct answers differ in cost.
* **Steeper cost reward:** a correct answer scores 0.1 at 2x gold's cost, 1 at gold's cost and 1.9 at half; wrong answers and the same shortcuts as before score 0.

15 iterations of 128 questions, about 3 hours (about 35 units); copies after 5 and 10 iterations are kept as their own runs. Then predictions for validation, dev and mini-dev of all three.

**Resumable:** after a disconnect, run the setup, install, config and staging cells, then the cell that was running.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# Starts from the RL model kept on validation (33_rl_evaluate).
INIT = json.load(open(f"{R}/rl_v1_choice/selection.json"))["winner"]
RUN = "rl_eff_v1"
# Snapshots after 5 and 10 of the 15 iterations (own runs), then the final model.
RUNS = [r for r in (f"{RUN}_it5", f"{RUN}_it10") if os.path.exists(f"{R}/{r}/adapter_meta.json")] + [RUN]
print("starts from:", INIT, "| run:", RUN)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### Stage the train databases (30 GB, a few minutes; skipped when already staged)

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### 1. Efficiency phase

In [ ]:
!python -m effsql.train rl --run {RUN} --init-from {INIT} --settings rl_eff

### 2. Predict validation, dev and mini-dev (final model and the two copies)

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# Starts from the RL model kept on validation (33_rl_evaluate).
INIT = json.load(open(f"{R}/rl_v1_choice/selection.json"))["winner"]
RUN = "rl_eff_v1"
# Snapshots after 5 and 10 of the 15 iterations (own runs), then the final model.
RUNS = [r for r in (f"{RUN}_it5", f"{RUN}_it10") if os.path.exists(f"{R}/{r}/adapter_meta.json")] + [RUN]
for r in RUNS:
    !python -m effsql.train predict --run {r} --splits val,dev,minidev

### Done: disconnect now, then run `35_rl_eff_evaluate` on a CPU runtime